In [15]:
!pip install keybert

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.2 MB/s eta 0:00:00
  Using cached requests-2.31.0-py3-none-any.whl.metadata (4.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
  Using cached urllib3-2.2.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached certifi-2024.2.2-py3-none-any.whl.metadata (2.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 9.8 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 11.3 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━

In [ ]:
from keybert import KeyBERT


In [14]:
def get_keywords():

		model = BertModel.from_pretrained('skt/kobert-base-v1')
    kw_model = KeyBERT(model)

    # 문장별로 문장, 전처리 문장
    sentences, pre_sentences = split_into_sentences(file_open("input.txt"))

    for id_num in range(1, len(sentences) + 1):
        keywords = kw_model.extract_keywords(pre_sentences[id_num - 1], keyphrase_ngram_range=(1, 1), stop_words=None, top_n=10)
        keywords = [item[0] for item in keywords]
        print("keyword : ", keywords)
        set_data(id_num - 1, sentences[id_num - 1], keywords)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 4)

In [ ]:
def noun_extractor(text):
    results = []
    kiwi = Kiwi()
    result = kiwi.analyze(text)
    for token, pos, _, _ in result[0][0]:
        if len(token) != 1 and pos.startswith('N') or pos.startswith('SL'):
            results.append(token)
    return results

In [4]:
from keybert import KeyBERT
from kiwipiepy import Kiwi
from transformers import BertModel
from elasticsearch import Elasticsearch

es = Elasticsearch("http://192.168.101.218:9200")
index_name = 'keyword_sentences'

def file_open(path):
    with open(path, "r") as f:
        text = f.read()
    return text

def noun_extractor(text):
    results = []
    kiwi = Kiwi()
    result = kiwi.analyze(text)
    for token, pos, _, _ in result[0][0]:
        if len(token) != 1 and pos.startswith('N') or pos.startswith('SL'):
            results.append(token)
    return results

def split_into_sentences(text):
    kiwi = Kiwi()
    str_list = kiwi.split_into_sents(text)
    keyword_list = []
    for id_num in range(1, len(str_list) + 1):
        str_list[id_num - 1] = str_list[id_num - 1].text
        keyword_list.append(preprocess(str_list[id_num - 1]))

    return str_list, keyword_list

def set_data(id_num, sentences, keyword):
    data = {"sequence": id_num, "text": sentences, "keyword": keyword}
    put(data, id_num)

def put(data, id_num):
    es.index(index=index_name, id=id_num, body=data)
    es.indices.refresh(index=index_name)

def preprocess(text):
    nouns = noun_extractor(text)

    return ' '.join(nouns)

def get_keywords(kw_model):
    # 문장별로 문장, 전처리 문장
    sentences, pre_sentences = split_into_sentences(file_open("input.txt"))

    for id_num in range(1, len(sentences) + 1):
        keywords = kw_model.extract_keywords(pre_sentences[id_num - 1], keyphrase_ngram_range=(1, 1), stop_words=None, top_n=10)
        keywords = [item[0] for item in keywords]
        set_data(id_num - 1, sentences[id_num - 1], keywords)

if __name__ == '__main__':

    # 모델 생성
    model = BertModel.from_pretrained('skt/kobert-base-v1')
    kw_model = KeyBERT(model)

    # input 키워드 생성
    get_keywords(kw_model)


ModuleNotFoundError: No module named 'keybert'